# 00 - Build the PPG library

Run this notebook **first**. It installs dependencies and writes every
module of the `ppg_breakhis` package to disk using `%%writefile`, so the
code stays modular and inspectable. Notebooks 01 and 02 then just import
these modules. No command line is used anywhere.

The corrected PPG math (fixed gate direction, monotone uncertainty gate,
rectified activation, `P = K x C` head, t-interval CIs) lives in
`models/ppg.py`.

## 1. Install dependencies

In [1]:
%pip install -q torch torchvision timm scikit-learn scipy Pillow tqdm

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.58.0 requires blinker<2,>=1.5.0, which is not installed.
automated-interpretability 0.0.3 requires httpx<0.28.0,>=0.27.0, but you have httpx 0.28.1 which is incompatible.
automated-interpretability 0.0.3 requires pytest<9.0.0,>=8.1.2, but you have pytest 9.1.1 which is incompatible.
automated-interpretability 0.0.3 requires tiktoken<0.7.0,>=0.6.0, but you have tiktoken 0.13.0 which is incompatible.


## 2. Create the package folders
`%%writefile` does not create parent directories, so we make them first.

In [2]:
import os
for d in ['ppg_breakhis', 'ppg_breakhis/data', 'ppg_breakhis/models',
          'ppg_breakhis/metrics', 'ppg_breakhis/engine']:
    os.makedirs(d, exist_ok=True)
    open(os.path.join(d, '__init__.py'), 'a').close()  # make it a package

import sys
sys.path.insert(0, os.path.abspath('ppg_breakhis'))  # so we can import modules
print('folders ready; import path set to', os.path.abspath('ppg_breakhis'))

folders ready; import path set to d:\Bureau\mythese\PhD_IA_XAI\07_Code\ppg_breakhis\ppg_breakhis


## 3. Write the modules
Each cell below writes one module. Edit any cell and re-run it to change
that module, then restart the kernel of notebook 01/02 to pick it up.

### `config.py`

In [3]:
%%writefile ppg_breakhis/config.py
"""Central configuration for the PPG-BreakHis experiments.

Every hyperparameter and ablation switch lives here so the rest of the code
stays clean. Override any field from the command line in train.py.
"""
from dataclasses import dataclass, field
from typing import Literal


@dataclass
class Config:
    # ---- data ----
    data_root: str = "./BreakHis_v1"          # folder that contains the images
    magnification: str = "200X"                # 40X | 100X | 200X | 400X
    image_size: int = 224
    val_frac: float = 0.10                      # fractions are PATIENT-level, not image-level
    test_frac: float = 0.20
    num_classes: int = 2                        # benign vs malignant
    class_names: tuple = ("benign", "malignant")

    # ---- backbone ----
    backbone: str = "swin_tiny_patch4_window7_224"
    pretrained: bool = True
    feature_stage: int = 3                      # 2 -> 1/16 (384ch), 3 -> 1/32 (768ch, matches D=768)

    # ---- prototypes ----
    protos_per_class: int = 10                  # K in the write-up; total P = K * num_classes
    temperature: float = 0.1                    # tau for the spatial softmax

    # ---- uncertainty ----
    mc_samples: int = 5                         # M (set to 1 to disable uncertainty)
    mc_dropout_p: float = 0.2                   # dropout applied to the feature map per MC pass

    # ---- gating (ablation switches) ----
    gate_direction: Literal["corrected", "original"] = "corrected"
    use_bernoulli_gate: bool = True             # the z gate
    use_uncertainty_gate: bool = True           # the g (HGLU-style) gate
    pooling: Literal["attention", "maxpool"] = "attention"
    head: Literal["fixed", "learnable"] = "fixed"
    beta: float = 5.0                           # gate steepness
    gamma: float = 0.1                          # gate threshold on uncertainty
    gumbel_tau: float = 0.5                     # relaxation temperature for z

    # ---- training ----
    epochs: int = 30
    batch_size: int = 16
    lr: float = 1e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    num_workers: int = 4
    seed: int = 0
    device: str = "cuda"

    # ---- logging ----
    out_dir: str = "./runs"
    exp_name: str = "ppg_swint"

    def total_protos(self) -> int:
        return self.protos_per_class * self.num_classes


Writing ppg_breakhis/config.py


### `data/breakhis.py`

In [4]:
%%writefile ppg_breakhis/data/breakhis.py
"""BreakHis dataset loader with a PATIENT-LEVEL split.

The single biggest mistake with BreakHis is splitting by image: the same
patient's slides then appear in both train and test, and accuracy becomes
meaningless. BreakHis filenames encode the patient/slide id, e.g.

    SOB_B_TA-14-4659-400-001.png
        |  |     |  |    |   |
        |  |     |  |    |   +-- sequence number
        |  |     |  |    +------ magnification (40 / 100 / 200 / 400)
        |  |     |  +----------- slide id      -> used as the PATIENT group
        |  |     +-------------- year
        |  +-------------------- tumor subtype (TA, DC, ...)
        +----------------------- class: B (benign) or M (malignant)

We group by "<year>-<slideid>" and split those groups, so no patient leaks.
"""
import os
import glob
import re
from typing import List, Tuple

import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit


_FNAME_RE = re.compile(
    r"SOB_(?P<cls>[BM])_[A-Za-z]+-(?P<year>\d+)-(?P<slide>[A-Za-z0-9]+)-(?P<mag>\d+)-\d+"
)


def _parse(fname: str):
    """Return (class_idx, patient_group, magnification) or None if unparseable."""
    m = _FNAME_RE.search(os.path.basename(fname))
    if m is None:
        return None
    cls = 0 if m.group("cls") == "B" else 1        # benign=0, malignant=1
    patient = f"{m.group('year')}-{m.group('slide')}"
    mag = m.group("mag") + "X"
    return cls, patient, mag


def scan_breakhis(root: str, magnification: str) -> Tuple[List[str], List[int], List[str]]:
    """Walk the BreakHis folder and collect paths/labels/patient-groups."""
    paths, labels, groups = [], [], []
    for p in glob.glob(os.path.join(root, "**", "*.png"), recursive=True):
        parsed = _parse(p)
        if parsed is None:
            continue
        cls, patient, mag = parsed
        if mag != magnification:
            continue
        paths.append(p)
        labels.append(cls)
        groups.append(patient)
    if not paths:
        raise RuntimeError(
            f"No {magnification} images found under {root}. "
            "Check data_root and that filenames follow the SOB_* convention."
        )
    return paths, labels, groups


def patient_level_split(paths, labels, groups, val_frac, test_frac, seed):
    """Two nested GroupShuffleSplits -> train / val / test with disjoint patients."""
    paths = np.array(paths)
    labels = np.array(labels)
    groups = np.array(groups)

    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, test_idx = next(gss_test.split(paths, labels, groups))

    val_ratio = val_frac / (1.0 - test_frac)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    tr_rel, va_rel = next(
        gss_val.split(paths[trainval_idx], labels[trainval_idx], groups[trainval_idx])
    )
    train_idx = trainval_idx[tr_rel]
    val_idx = trainval_idx[va_rel]

    # sanity: no patient overlap
    for a, b in [(train_idx, val_idx), (train_idx, test_idx), (val_idx, test_idx)]:
        assert set(groups[a]).isdisjoint(set(groups[b])), "patient leak between splits!"

    return {
        "train": (paths[train_idx], labels[train_idx]),
        "val": (paths[val_idx], labels[val_idx]),
        "test": (paths[test_idx], labels[test_idx]),
    }


_MEAN = (0.485, 0.456, 0.406)
_STD = (0.229, 0.224, 0.225)


def build_transforms(image_size: int, train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.1, 0.1, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])


class BreakHisDataset(Dataset):
    def __init__(self, paths, labels, image_size, train):
        self.paths = list(paths)
        self.labels = list(labels)
        self.tf = build_transforms(image_size, train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        x = self.tf(img)
        y = int(self.labels[i])
        return x, y


def make_datasets(cfg):
    """Convenience: returns train/val/test BreakHisDataset objects from a Config."""
    paths, labels, groups = scan_breakhis(cfg.data_root, cfg.magnification)
    split = patient_level_split(paths, labels, groups,
                                cfg.val_frac, cfg.test_frac, cfg.seed)
    ds = {
        name: BreakHisDataset(p, y, cfg.image_size, train=(name == "train"))
        for name, (p, y) in split.items()
    }
    return ds


Writing ppg_breakhis/data/breakhis.py


### `models/backbone.py`

In [5]:
%%writefile ppg_breakhis/models/backbone.py
"""Swin-Tiny backbone that returns a spatial feature map [B, C, H, W].

timm's `features_only` output layout differs across versions (some return
channels-last NHWC for Swin). We detect the channel dim and normalise to NCHW
so the rest of the code never has to care.
"""
import timm
import torch
import torch.nn as nn


class SwinFeatureExtractor(nn.Module):
    def __init__(self, name="swin_tiny_patch4_window7_224", pretrained=True, stage=3):
        super().__init__()
        self.backbone = timm.create_model(
            name, pretrained=pretrained, features_only=True, out_indices=(stage,)
        )
        self.out_channels = self.backbone.feature_info.channels()[-1]

    @staticmethod
    def _to_nchw(x, c):
        # already NCHW
        if x.dim() == 4 and x.shape[1] == c:
            return x
        # NHWC -> NCHW
        if x.dim() == 4 and x.shape[-1] == c:
            return x.permute(0, 3, 1, 2).contiguous()
        raise RuntimeError(f"Unexpected feature shape {tuple(x.shape)} for C={c}")

    def forward(self, x):
        feats = self.backbone(x)[-1]
        return self._to_nchw(feats, self.out_channels)   # [B, C, H, W]


Writing ppg_breakhis/models/backbone.py


### `models/ppg.py`

In [6]:
%%writefile ppg_breakhis/models/ppg.py
"""Probabilistic Prototype Gating (PPG) layer and PPG-SwinT model.

This implements the CORRECTED design discussed in review:

  1. cosine similarity between prototypes and every spatial location
  2. temperature-scaled spatial softmax  -> attention  (or hard max, ablation)
  3. M MC-dropout passes -> mean mu and UNBIASED variance -> uncertainty u = sigma
  4. gate that CLOSES on high uncertainty:  p = sigmoid(beta * (gamma - u))
        (the 'original' ablation flips the sign, reproducing the buggy version)
  5. Gumbel-sigmoid (binary concrete) sample z  (differentiable)
  6. TRUE multiplicative uncertainty gate g = sigmoid(b - softplus(theta) . u),
        monotonically decreasing in u by construction (non-negative weights)
  7. rectified mean so activations are non-negative:  a = z * g * relu(mu)
  8. head:  y = W a,  with P = K * C total prototypes (fixed block weights or learnable)

Set mc_samples=1 to disable uncertainty (u=0, gates open) for the ablation.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def gumbel_sigmoid(logit_p, tau, training):
    """Binary-concrete relaxation of Bernoulli(sigmoid(logit_p)).

    z = sigmoid((logit_p + L) / tau), L ~ Logistic(0,1) = log U - log(1-U).
    At eval we drop the noise and return the expected (deterministic) gate.
    """
    if not training:
        return torch.sigmoid(logit_p)
    u = torch.rand_like(logit_p).clamp_(1e-6, 1 - 1e-6)
    noise = torch.log(u) - torch.log1p(-u)          # logistic noise = g1 - g2
    return torch.sigmoid((logit_p + noise) / tau)


class PPGLayer(nn.Module):
    def __init__(self, cfg, feat_dim):
        super().__init__()
        self.cfg = cfg
        P = cfg.total_protos()
        self.P = P
        self.K = cfg.protos_per_class
        self.C = cfg.num_classes

        # prototypes: [P, feat_dim]
        self.prototypes = nn.Parameter(torch.randn(P, feat_dim) * 0.02)
        self.log_tau = nn.Parameter(torch.tensor(math.log(cfg.temperature)))

        # uncertainty gate g: non-negative weights via softplus(theta)
        self.gate_theta = nn.Parameter(torch.zeros(P, P))
        self.gate_bias = nn.Parameter(torch.ones(P) * 2.0)   # start with gates open

        self.dropout = nn.Dropout2d(cfg.mc_dropout_p)

        # classification head
        if cfg.head == "fixed":
            W = torch.zeros(self.C, P)
            for c in range(self.C):
                W[c, c * self.K:(c + 1) * self.K] = 1.0 / self.K
            self.register_buffer("W_head", W)
            self.head = None
        else:
            self.head = nn.Linear(P, self.C, bias=False)

    # ---- one stochastic scoring pass over a feature map ----
    def _score_once(self, feat):
        """feat: [B, C, H, W] -> (score [B,P], alpha [B,P,H,W])."""
        B, Cc, Hh, Ww = feat.shape
        f = self.dropout(feat)                                   # MC-dropout perturbation
        f = F.normalize(f, dim=1)                                # unit feature vectors
        p = F.normalize(self.prototypes, dim=1)                  # unit prototypes [P, C]
        # cosine similarity map S: [B, P, H, W]
        S = torch.einsum("pc,bchw->bphw", p, f)

        if self.cfg.pooling == "maxpool":
            score = S.flatten(2).max(dim=2).values               # [B, P]
            alpha = torch.zeros_like(S)                          # placeholder for viz
            idx = S.flatten(2).argmax(dim=2)
            alpha.flatten(2).scatter_(2, idx.unsqueeze(-1), 1.0)
        else:
            tau = self.log_tau.exp().clamp(min=1e-3)
            alpha = F.softmax(S.flatten(2) / tau, dim=2).view_as(S)  # [B,P,H,W], sums to 1
            score = (alpha * S).flatten(2).sum(dim=2)                # attention-weighted mean

        return score, alpha

    def forward(self, feat):
        """feat: [B, C, H, W]. Returns dict with logits, mu, sigma, alpha."""
        M = self.cfg.mc_samples
        # run M stochastic passes (dropout differs each time)
        scores, alphas = [], []
        for _ in range(max(M, 1)):
            s, a = self._score_once(feat)
            scores.append(s)
            alphas.append(a)
        scores = torch.stack(scores, dim=0)      # [M, B, P]
        alpha = torch.stack(alphas, dim=0).mean(0)  # [B, P, H, W] for visualisation

        mu = scores.mean(dim=0)                   # [B, P]
        if M > 1:
            sigma = scores.var(dim=0, unbiased=True).clamp_min(1e-12).sqrt()  # [B, P]
        else:
            sigma = torch.zeros_like(mu)          # no uncertainty available

        u = sigma                                 # epistemic proxy

        # ---- Bernoulli gate z ----
        if self.cfg.use_bernoulli_gate and M > 1:
            if self.cfg.gate_direction == "corrected":
                gate_logit = self.cfg.beta * (self.cfg.gamma - u)   # closes on high u
            else:  # 'original' -> the buggy sign that OPENS on high u
                gate_logit = self.cfg.beta * (u - self.cfg.gamma)
            z = gumbel_sigmoid(gate_logit, self.cfg.gumbel_tau, self.training)
        else:
            z = torch.ones_like(mu)

        # ---- uncertainty gate g (monotone decreasing in u) ----
        if self.cfg.use_uncertainty_gate and M > 1:
            W_tilde = F.softplus(self.gate_theta)                  # [P, P] >= 0
            g = torch.sigmoid(self.gate_bias - u @ W_tilde.t())    # [B, P]
        else:
            g = torch.ones_like(mu)

        a = z * g * F.relu(mu)                    # [B, P]

        if self.head is None:
            logits = a @ self.W_head.t()          # fixed positive block weights
        else:
            logits = self.head(a)

        return {"logits": logits, "mu": mu, "sigma": sigma, "alpha": alpha, "a": a}


class PPGSwinT(nn.Module):
    """Backbone + PPG head."""
    def __init__(self, cfg, backbone):
        super().__init__()
        self.backbone = backbone
        self.ppg = PPGLayer(cfg, feat_dim=backbone.out_channels)

    def forward(self, x):
        feat = self.backbone(x)
        return self.ppg(feat)


def prototype_confidence_interval(mu, sigma, M, level=0.95):
    """Per-prototype CI using a t-interval (correct for small M), not 1.96*z.

    Returns (low, high) for the MEAN activation. For M=5, t_{0.975,4}=2.776.
    """
    from scipy import stats
    if M <= 1:
        return mu, mu
    t = stats.t.ppf(0.5 + level / 2.0, df=M - 1)
    half = t * sigma / math.sqrt(M)
    return mu - half, mu + half


Writing ppg_breakhis/models/ppg.py


### `models/baselines.py`

In [7]:
%%writefile ppg_breakhis/models/baselines.py
"""Baselines. Each isolates one of PPG's two claims.

  PlainSwin        : no prototypes, no uncertainty  -> is any machinery needed?
  DeterministicProto: prototypes + hard max, no uncertainty -> does the
                       PROBABILISTIC part beat plain prototypes?
  MCDropoutViT     : uncertainty but no prototypes -> does the PROTOTYPE part
                       add anything over plain uncertainty?

All return the same dict shape as PPGSwinT for a uniform training/eval loop.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class PlainSwin(nn.Module):
    def __init__(self, cfg, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_channels, cfg.num_classes)

    def forward(self, x):
        feat = self.backbone(x)                       # [B, C, H, W]
        pooled = feat.mean(dim=(2, 3))                # global average pool
        return {"logits": self.head(pooled)}


class DeterministicProto(nn.Module):
    """ProtoPNet / HierViT-style: cosine prototypes + hard max, learnable head."""
    def __init__(self, cfg, backbone):
        super().__init__()
        self.backbone = backbone
        P = cfg.total_protos()
        self.prototypes = nn.Parameter(torch.randn(P, backbone.out_channels) * 0.02)
        self.head = nn.Linear(P, cfg.num_classes, bias=False)

    def forward(self, x):
        feat = F.normalize(self.backbone(x), dim=1)
        p = F.normalize(self.prototypes, dim=1)
        S = torch.einsum("pc,bchw->bphw", p, feat)    # [B, P, H, W]
        act = S.flatten(2).max(dim=2).values          # hard max pooling
        return {"logits": self.head(F.relu(act)), "alpha": S}


class MCDropoutViT(nn.Module):
    """Backbone + dropout + linear head; M passes at eval for uncertainty."""
    def __init__(self, cfg, backbone):
        super().__init__()
        self.cfg = cfg
        self.backbone = backbone
        self.drop = nn.Dropout(cfg.mc_dropout_p)
        self.head = nn.Linear(backbone.out_channels, cfg.num_classes)

    def _once(self, feat):
        pooled = self.drop(feat.mean(dim=(2, 3)))
        return self.head(pooled)

    def forward(self, x):
        feat = self.backbone(x)
        M = self.cfg.mc_samples if not self.training else 1
        logits = torch.stack([self._once(feat) for _ in range(max(M, 1))], 0).mean(0)
        return {"logits": logits}


def build_model(cfg, backbone):
    """Factory used by train.py."""
    name = cfg.exp_name.lower()
    if name.startswith("plain"):
        return PlainSwin(cfg, backbone)
    if name.startswith("proto") or name.startswith("hiervit"):
        return DeterministicProto(cfg, backbone)
    if name.startswith("mcdropout"):
        return MCDropoutViT(cfg, backbone)
    # default -> the proposed model
    from .ppg import PPGSwinT
    return PPGSwinT(cfg, backbone)


Writing ppg_breakhis/models/baselines.py


### `metrics/classification.py`

In [8]:
%%writefile ppg_breakhis/metrics/classification.py
"""Accuracy, AUC-ROC and macro-F1."""
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score


def classification_metrics(probs, labels, num_classes):
    """probs: [N, C] softmax probs. labels: [N] ints."""
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    preds = probs.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    try:
        if num_classes == 2:
            auc = roc_auc_score(labels, probs[:, 1])
        else:
            auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")   # happens if a class is absent from a tiny split

    return {"accuracy": acc, "auc": auc, "f1": f1}


Writing ppg_breakhis/metrics/classification.py


### `metrics/calibration.py`

In [9]:
%%writefile ppg_breakhis/metrics/calibration.py
"""Calibration: Expected Calibration Error (ECE) and Negative Log-Likelihood.

This is where the uncertainty claim is actually tested - keep it in every run.
"""
import numpy as np


def expected_calibration_error(probs, labels, n_bins=15):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(np.float64)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(labels)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        acc_bin = correct[mask].mean()
        conf_bin = conf[mask].mean()
        ece += (mask.sum() / N) * abs(acc_bin - conf_bin)
    return float(ece)


def negative_log_likelihood(probs, labels, eps=1e-12):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    p_true = probs[np.arange(len(labels)), labels]
    return float(-np.log(np.clip(p_true, eps, 1.0)).mean())


def calibration_metrics(probs, labels):
    return {
        "ece": expected_calibration_error(probs, labels),
        "nll": negative_log_likelihood(probs, labels),
    }


Writing ppg_breakhis/metrics/calibration.py


### `metrics/pointing_game.py`

In [10]:
%%writefile ppg_breakhis/metrics/pointing_game.py
"""Pointing Game interpretability metric.

IMPORTANT CAVEAT FOR BreakHis: this metric needs a ground-truth localisation
(a bounding box or segmentation mask of the pathology). BreakHis provides only
image-level benign/malignant labels - it has NO such boxes. So Pointing Game is
*not directly runnable on BreakHis* out of the box.

Options:
  (a) skip Pointing Game for the BreakHis pilot (report only acc/AUC/ECE), or
  (b) evaluate it on a dataset that ships masks (e.g. ISIC provides lesion
      segmentations), or
  (c) have an expert annotate a small held-out subset of BreakHis.

The implementation below is generic: give it upsampled prototype attention maps
and binary masks and it computes the hit rate. It is here so the code is ready
the moment masks are available.
"""
import numpy as np
import torch
import torch.nn.functional as F


def upsample_attention(alpha, size):
    """alpha: [B, P, h, w] -> [B, P, H, W] bilinearly upsampled to `size`=(H,W)."""
    return F.interpolate(alpha, size=size, mode="bilinear", align_corners=False)


def pointing_game(alpha_upsampled, masks, class_of_proto, labels):
    """Hit if the argmax location of the most-activated prototype for the true
    class falls inside the ground-truth mask.

    alpha_upsampled : [B, P, H, W] tensor
    masks           : [B, H, W] binary tensor (1 = pathology region)
    class_of_proto  : [P] long tensor mapping prototype -> class index
    labels          : [B] long tensor of true classes
    returns         : hit rate in [0, 1]
    """
    B, P, H, W = alpha_upsampled.shape
    hits = 0
    for b in range(B):
        cls = int(labels[b])
        proto_idx = (class_of_proto == cls).nonzero(as_tuple=True)[0]
        if len(proto_idx) == 0:
            continue
        # pick the prototype (of the true class) with the strongest peak
        sub = alpha_upsampled[b, proto_idx]              # [k, H, W]
        peaks = sub.flatten(1).max(dim=1).values
        best = proto_idx[int(peaks.argmax())]
        flat = alpha_upsampled[b, best].flatten().argmax()
        y, x = divmod(int(flat), W)
        if masks[b, y, x] > 0.5:
            hits += 1
    return hits / max(B, 1)


def class_of_proto_tensor(protos_per_class, num_classes, device="cpu"):
    return torch.arange(num_classes, device=device).repeat_interleave(protos_per_class)


Writing ppg_breakhis/metrics/pointing_game.py


### `engine/trainer.py`

In [11]:
%%writefile ppg_breakhis/engine/trainer.py
"""Training / evaluation loops shared by every model."""
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from metrics.classification import classification_metrics
from metrics.calibration import calibration_metrics


def make_loaders(datasets, cfg):
    def dl(ds, shuffle):
        return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle,
                          num_workers=cfg.num_workers, pin_memory=True, drop_last=False)
    return {
        "train": dl(datasets["train"], True),
        "val": dl(datasets["val"], False),
        "test": dl(datasets["test"], False),
    }


def cosine_warmup(step, total, warmup, base_lr):
    if step < warmup:
        return base_lr * (step + 1) / max(warmup, 1)
    prog = (step - warmup) / max(total - warmup, 1)
    return 0.5 * base_lr * (1 + math.cos(math.pi * prog))


@torch.no_grad()
def evaluate(model, loader, cfg):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(cfg.device)
        out = model(x)
        probs = F.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    m = {}
    m.update(classification_metrics(probs, labels, cfg.num_classes))
    m.update(calibration_metrics(probs, labels))
    return m


def train(model, datasets, cfg, verbose=True):
    loaders = make_loaders(datasets, cfg)
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.CrossEntropyLoss()

    total_steps = cfg.epochs * len(loaders["train"])
    warmup_steps = cfg.warmup_epochs * len(loaders["train"])
    step = 0
    best_val_auc, best_state = -1.0, None

    for epoch in range(cfg.epochs):
        model.train()
        running = 0.0
        for x, y in loaders["train"]:
            x, y = x.to(cfg.device), y.to(cfg.device)
            for g in opt.param_groups:
                g["lr"] = cosine_warmup(step, total_steps, warmup_steps, cfg.lr)
            out = model(x)
            loss = criterion(out["logits"], y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
            step += 1

        val = evaluate(model, loaders["val"], cfg)
        if verbose:
            print(f"[{cfg.exp_name}] epoch {epoch+1:02d}/{cfg.epochs} "
                  f"loss={running/len(datasets['train']):.4f} "
                  f"val_acc={val['accuracy']:.4f} val_auc={val['auc']:.4f} "
                  f"val_ece={val['ece']:.4f}")
        if val["auc"] > best_val_auc:
            best_val_auc = val["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    test = evaluate(model, loaders["test"], cfg)
    if verbose:
        print(f"[{cfg.exp_name}] TEST  acc={test['accuracy']:.4f} auc={test['auc']:.4f} "
              f"f1={test['f1']:.4f} ece={test['ece']:.4f} nll={test['nll']:.4f}")
    return model, test


Writing ppg_breakhis/engine/trainer.py


### `train.py`

In [12]:
%%writefile ppg_breakhis/train.py
"""Train a single model on BreakHis.

Examples
--------
# the proposed model
python train.py --exp_name ppg_swint

# baselines (exp_name prefix selects the architecture)
python train.py --exp_name plain
python train.py --exp_name proto          # deterministic prototypes (HierViT-style)
python train.py --exp_name mcdropout

# a quick smoke test on CPU without touching real data
python train.py --smoke
"""
import argparse
import dataclasses
import json
import os
import random

import numpy as np
import torch

from config import Config
from models.backbone import SwinFeatureExtractor
from models.baselines import build_model
from engine.trainer import train


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_args():
    p = argparse.ArgumentParser()
    for f in dataclasses.fields(Config):
        if f.type is bool:
            p.add_argument(f"--{f.name}", type=lambda s: s.lower() in ("1", "true", "yes"),
                           default=None)
        else:
            p.add_argument(f"--{f.name}", type=type(f.default), default=None)
    p.add_argument("--smoke", action="store_true", help="tiny run on synthetic data")
    return p.parse_args()


def apply_overrides(cfg, args):
    for f in dataclasses.fields(Config):
        v = getattr(args, f.name)
        if v is not None:
            setattr(cfg, f.name, v)
    return cfg


def smoke_test(cfg):
    """Verify the whole model wiring on random tensors (no data, no GPU needed)."""
    cfg.device = "cpu"
    cfg.pretrained = False
    backbone = SwinFeatureExtractor(cfg.backbone, pretrained=False, stage=cfg.feature_stage)
    model = build_model(cfg, backbone).to(cfg.device)
    x = torch.randn(2, 3, cfg.image_size, cfg.image_size)
    out = model(x)
    print("smoke OK -> logits", tuple(out["logits"].shape),
          "| keys:", list(out.keys()))


def main():
    args = parse_args()
    cfg = apply_overrides(Config(), args)
    set_seed(cfg.seed)

    if args.smoke:
        smoke_test(cfg)
        return

    if not torch.cuda.is_available():
        cfg.device = "cpu"

    from data.breakhis import make_datasets
    datasets = make_datasets(cfg)
    print({k: len(v) for k, v in datasets.items()})

    backbone = SwinFeatureExtractor(cfg.backbone, pretrained=cfg.pretrained,
                                    stage=cfg.feature_stage)
    model = build_model(cfg, backbone)
    model, test_metrics = train(model, datasets, cfg)

    os.makedirs(cfg.out_dir, exist_ok=True)
    with open(os.path.join(cfg.out_dir, f"{cfg.exp_name}_seed{cfg.seed}.json"), "w") as fh:
        json.dump({"config": dataclasses.asdict(cfg), "test": test_metrics}, fh, indent=2)


if __name__ == "__main__":
    main()


Writing ppg_breakhis/train.py


### `run_ablations.py`

In [13]:
%%writefile ppg_breakhis/run_ablations.py
"""Run the ablation grid + baselines over several seeds and print a summary table.

Each ablation isolates one design choice flagged in review. Run:

    python run_ablations.py --seeds 0 1 2
"""
import argparse
import copy
import dataclasses
import json
import os

import numpy as np
import torch

from config import Config
from models.backbone import SwinFeatureExtractor
from models.baselines import build_model
from engine.trainer import train
from train import set_seed
from data.breakhis import make_datasets


# name -> dict of Config overrides. exp_name prefix picks the architecture.
ABLATIONS = {
    # --- baselines ---
    "plain_swin":        dict(exp_name="plain"),
    "proto_det":         dict(exp_name="proto"),
    "mcdropout_vit":     dict(exp_name="mcdropout"),
    # --- proposed model, full ---
    "ppg_full":          dict(exp_name="ppg_swint"),
    # --- ablations on the proposed model ---
    "ppg_gate_original": dict(exp_name="ppg_gateorig", gate_direction="original"),
    "ppg_no_g_gate":     dict(exp_name="ppg_nog", use_uncertainty_gate=False),
    "ppg_no_z_gate":     dict(exp_name="ppg_noz", use_bernoulli_gate=False),
    "ppg_maxpool":       dict(exp_name="ppg_maxpool", pooling="maxpool"),
    "ppg_M1":            dict(exp_name="ppg_m1", mc_samples=1),
    "ppg_M3":            dict(exp_name="ppg_m3", mc_samples=3),
    "ppg_learnable_head":dict(exp_name="ppg_learnhead", head="learnable"),
}


def run_one(base_cfg, overrides, datasets, seed):
    cfg = copy.deepcopy(base_cfg)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    cfg.seed = seed
    set_seed(seed)
    backbone = SwinFeatureExtractor(cfg.backbone, pretrained=cfg.pretrained,
                                    stage=cfg.feature_stage)
    model = build_model(cfg, backbone)
    _, test = train(model, datasets, cfg, verbose=False)
    return test


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--data_root", type=str, default=None)
    args = ap.parse_args()

    base = Config()
    if args.data_root:
        base.data_root = args.data_root
    if not torch.cuda.is_available():
        base.device = "cpu"

    # build datasets once (patient split is seeded by base.seed; keep fixed across
    # ablations so differences come from the METHOD, not the split)
    datasets = make_datasets(base)

    results = {name: [] for name in ABLATIONS}
    for name, ov in ABLATIONS.items():
        for seed in args.seeds:
            m = run_one(base, ov, datasets, seed)
            results[name].append(m)
            print(f"{name:20s} seed={seed} acc={m['accuracy']:.3f} "
                  f"auc={m['auc']:.3f} ece={m['ece']:.3f}")

    # summary: mean +/- std
    print("\n=== SUMMARY (mean +/- std over seeds) ===")
    header = f"{'ablation':20s} {'acc':>14s} {'auc':>14s} {'ece':>14s}"
    print(header)
    print("-" * len(header))
    summary = {}
    for name, runs in results.items():
        row = {}
        line = f"{name:20s}"
        for key in ("accuracy", "auc", "ece"):
            vals = np.array([r[key] for r in runs], dtype=float)
            row[key] = (float(np.nanmean(vals)), float(np.nanstd(vals)))
            line += f" {row[key][0]:.3f}+/-{row[key][1]:.3f}"
        summary[name] = row
        print(line)

    os.makedirs(base.out_dir, exist_ok=True)
    with open(os.path.join(base.out_dir, "ablation_summary.json"), "w") as fh:
        json.dump({"raw": results, "summary": summary}, fh, indent=2)


if __name__ == "__main__":
    main()


Writing ppg_breakhis/run_ablations.py


## 4. Smoke test (no data, no GPU, no internet)
Verifies the whole model wiring, that gradients flow, that the confidence
interval is valid, and that the corrected gate **decreases** as
uncertainty rises. Uses a tiny fake backbone so it runs instantly.

In [14]:
import torch, torch.nn as nn, torch.nn.functional as F
from config import Config
from models.baselines import build_model
from models.ppg import PPGSwinT, prototype_confidence_interval

class FakeBackbone(nn.Module):
    out_channels = 768
    def __init__(self):
        super().__init__(); self.c = nn.Conv2d(3, 768, 16, 16)
    def forward(self, x):
        return torch.relu(self.c(x))  # [B, 768, 14, 14]

def check(exp, **ov):
    cfg = Config(pretrained=False, device='cpu', **ov); cfg.exp_name = exp
    m = build_model(cfg, FakeBackbone())
    x = torch.randn(4, 3, 224, 224)
    m.train(); out = m(x)
    nn.CrossEntropyLoss()(out['logits'], torch.randint(0, 2, (4,))).backward()
    m.eval(); _ = m(x)
    print(f"{exp:14s} logits{tuple(out['logits'].shape)} keys={list(out.keys())}")

for name, ov in [('ppg_swint', {}), ('ppg_orig', dict(gate_direction='original')),
                 ('ppg_nog', dict(use_uncertainty_gate=False)),
                 ('ppg_max', dict(pooling='maxpool')), ('ppg_m1', dict(mc_samples=1)),
                 ('plain', {}), ('proto', {}), ('mcdropout', {})]:
    check(name, **ov)

cfg = Config(pretrained=False); m = PPGSwinT(cfg, FakeBackbone()).eval()
out = m(torch.randn(4, 3, 224, 224))
lo, hi = prototype_confidence_interval(out['mu'], out['sigma'], cfg.mc_samples)
print('CI valid (hi>=lo):', bool((hi >= lo).all()))
Wt = F.softplus(m.ppg.gate_theta)
g_lo = torch.sigmoid(m.ppg.gate_bias - torch.zeros(1, m.ppg.P) @ Wt.t()).mean().item()
g_hi = torch.sigmoid(m.ppg.gate_bias - (torch.ones(1, m.ppg.P)*2) @ Wt.t()).mean().item()
print(f'gate decreases with uncertainty: g(low)={g_lo:.3f} >= g(high)={g_hi:.3f}')
print('SMOKE TEST PASSED')

c:\Users\youssef\.conda\envs\xai-vit\Lib\site-packages\torch\cuda\__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


ppg_swint      logits(4, 2) keys=['logits', 'mu', 'sigma', 'alpha', 'a']
ppg_orig       logits(4, 2) keys=['logits', 'mu', 'sigma', 'alpha', 'a']
ppg_nog        logits(4, 2) keys=['logits', 'mu', 'sigma', 'alpha', 'a']
ppg_max        logits(4, 2) keys=['logits', 'mu', 'sigma', 'alpha', 'a']
ppg_m1         logits(4, 2) keys=['logits', 'mu', 'sigma', 'alpha', 'a']
plain          logits(4, 2) keys=['logits']
proto          logits(4, 2) keys=['logits', 'alpha']
mcdropout      logits(4, 2) keys=['logits']
CI valid (hi>=lo): True
gate decreases with uncertainty: g(low)=0.881 >= g(high)=0.000
SMOKE TEST PASSED
